# Pictures to video clips, free (LTX-Video on a Colab GPU)

Turns still pictures (the paintings or clay scenes from an explainroo video) into short video clips with
**LTX-Video 0.9.5** (2 billion parameters, open weights, uses the picture as the first frame). Free Colab T4 GPU.

**What you do**
1. On your computer: `node scripts/colab-clips.mjs pack videos/<project> --shots 3,7,12` makes a folder with those pictures and a `prompts.json`.
2. Put that folder in your Google Drive (default name `clay-clips`, in "My Drive").
3. Here: **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. The first run makes ONE clip as a test and prints how long it took.
4. When the test looks good, set `FIRST_N` to `0` (all pictures) and run the last cells again.
5. Clips appear in Drive under `clay-clips/clips/`. Copy them to your computer and run `node scripts/colab-clips.mjs unpack videos/<project> <folder>`.

**Know before you start**
- Free Colab gives about 15 GB of GPU memory and about 12 GB of system RAM, no guarantee of a GPU, and ends sessions after a few hours or when idle.
- Measured on a free T4: a **2 second** clip at 480 x 864 works; **4 seconds ran out of GPU memory** (the notebook then falls back to 2 seconds by itself, after wasting some minutes, so keep SECONDS at 2). Motion is gentle: a camera push, smoke, fire; not big action.
- Run once on a real free T4: one 2 second clip took about 8 minutes (including a failed first attempt at 4 seconds), after about 15 minutes of one-time downloads (10 GB text encoder, then the video model). If a cell fails, copy the error to Claude.

In [ ]:
#@title 1. Install (about 1 minute)
%%capture
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg

In [ ]:
#@title 2. Check the GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "nvidia-smi not found")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Menu: Runtime > Change runtime type > T4 GPU, then Run all again.")
cap = torch.cuda.get_device_capability()
print(f"GPU compute capability {cap[0]}.{cap[1]}:", "bfloat16 is native" if cap[0] >= 8 else "bfloat16 is emulated on this GPU (it works, but slower)")

In [ ]:
#@title 3. Settings
USE_DRIVE = True  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER = "/content/drive/MyDrive/clay-clips"  #@param {type:"string"}
else:
    # no Google Drive: upload the pictures and prompts.json with the Files panel on the left; they land in /content
    FOLDER = "/content"
import os
os.makedirs(FOLDER, exist_ok=True)
ORIENTATION = "portrait"  #@param ["portrait", "landscape"]
SIZE = "standard"  #@param ["draft", "standard"]
SECONDS = 2  #@param {type:"slider", min:2, max:6, step:1}
STEPS = 30  #@param {type:"slider", min:15, max:50, step:5}
FIRST_N = 1  #@param {type:"integer"}
SEED = 7  #@param {type:"integer"}
DEFAULT_PROMPT = "Subtle natural motion, smoke and embers drifting, flickering light, a slow smooth camera move, cinematic, painterly."  #@param {type:"string"}

# sizes must be multiples of 32; frames must be 8 x n + 1 (24 x seconds + 1 always is)
SIZES = {"draft": (352, 640), "standard": (480, 864)}
W, H = SIZES[SIZE]
SMALL_W, SMALL_H = SIZES["draft"]
if ORIENTATION == "landscape":
    W, H, SMALL_W, SMALL_H = H, W, SMALL_H, SMALL_W
FRAMES = 24 * SECONDS + 1
print(f"{W} x {H}, {FRAMES} frames ({SECONDS} s at 24 fps), {STEPS} steps")

In [ ]:
#@title 4. Read the pictures and the prompts
import glob, gc, json, os, time
from PIL import Image, ImageOps

EXT = (".png", ".jpg", ".jpeg", ".webp")
pics = sorted(p for p in glob.glob(os.path.join(FOLDER, "*")) if p.lower().endswith(EXT) and ".depth" not in os.path.basename(p))
OUT = os.path.join(FOLDER, "clips")
os.makedirs(OUT, exist_ok=True)
stem = lambda p: os.path.splitext(os.path.basename(p))[0]
prompts_file = os.path.join(FOLDER, "prompts.json")
prompts = json.load(open(prompts_file)) if os.path.exists(prompts_file) else {}

todo = [p for p in pics if not os.path.exists(os.path.join(OUT, stem(p) + ".mp4"))]
if FIRST_N > 0:
    todo = todo[:FIRST_N]
print(f"{len(pics)} pictures in {FOLDER}; {len(todo)} to make this run; {len(prompts)} prompts from prompts.json")
if not pics:
    raise RuntimeError(f"No pictures found in {FOLDER}. Upload the folder made by 'colab-clips.mjs pack' to your Drive first.")
if not todo:
    raise RuntimeError("Nothing to do: every picture already has a clip in the clips folder.")

In [ ]:
#@title 5. Write the prompts into numbers first (frees 9 GB of memory afterwards)
# The text encoder (T5) is about 9.5 GB, too much to keep next to the video model on the free tier. So it runs alone,
# turns every prompt into numbers, and is then thrown away.
import torch
from diffusers import AutoModel, LTXConditionPipeline
from diffusers.hooks import apply_group_offloading
from diffusers.utils import export_to_video

MODEL = "Lightricks/LTX-Video-0.9.5"
NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted, text, watermark"

te_pipe = LTXConditionPipeline.from_pretrained(MODEL, transformer=None, vae=None, dtype=torch.bfloat16).to("cuda")
embeds = {}
with torch.no_grad():
    for p in todo:
        text = prompts.get(stem(p), DEFAULT_PROMPT)
        pe, pm, ne, nm = te_pipe.encode_prompt(prompt=text, negative_prompt=NEGATIVE, do_classifier_free_guidance=True, device="cuda")
        embeds[stem(p)] = [t.cpu() for t in (pe, pm, ne, nm)]
del te_pipe
gc.collect()
torch.cuda.empty_cache()
print("prompts ready:", len(embeds))

In [ ]:
#@title 6. Load the video model (about 2 to 5 minutes the first time)
transformer = AutoModel.from_pretrained(MODEL, subfolder="transformer", dtype=torch.bfloat16)
# the weights sit in 8-bit and are widened one layer at a time while they are used
transformer.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.bfloat16)
pipe = LTXConditionPipeline.from_pretrained(MODEL, transformer=transformer, text_encoder=None, tokenizer=None, dtype=torch.bfloat16)

onload, offload = torch.device("cuda"), torch.device("cpu")
pipe.transformer.enable_group_offload(onload_device=onload, offload_device=offload, offload_type="leaf_level", use_stream=True)
apply_group_offloading(pipe.vae, onload_device=onload, offload_type="leaf_level")
try:
    pipe.vae.enable_tiling()
except Exception as e:
    print("vae tiling not available:", e)
print("model ready; GPU memory in use:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

In [ ]:
#@title 7. Make the clips
def fit(path, w, h):
    return ImageOps.fit(Image.open(path).convert("RGB"), (w, h), method=Image.LANCZOS, centering=(0.5, 0.5))

def generate(name, path, frames, w, h):
    pe, pm, ne, nm = [t.to("cuda") for t in embeds[name]]
    result = pipe(
        prompt_embeds=pe, prompt_attention_mask=pm, negative_prompt_embeds=ne, negative_prompt_attention_mask=nm,
        image=fit(path, w, h), width=w, height=h, num_frames=frames, frame_rate=24,
        num_inference_steps=STEPS, guidance_scale=3.0, decode_timestep=0.03, decode_noise_scale=0.025,
        generator=torch.Generator("cpu").manual_seed(SEED),
    )
    return result.frames[0]

# if the GPU runs out of memory, try a shorter clip, then a shorter and smaller one
ATTEMPTS = [(FRAMES, W, H)]
if FRAMES > 49:
    ATTEMPTS.append((49, W, H))
ATTEMPTS.append((49, SMALL_W, SMALL_H))

made = []
for p in todo:
    name = stem(p)
    target = os.path.join(OUT, name + ".mp4")
    started = time.time()
    for frames, w, h in ATTEMPTS:
        try:
            video = generate(name, p, frames, w, h)
            tmp = target + ".part.mp4"
            export_to_video(video, tmp, fps=24)
            os.replace(tmp, target)
            made.append(name)
            print(f"{name}: {frames} frames at {w} x {h} in {(time.time() - started) / 60:.1f} min -> {target}")
            break
        except torch.cuda.OutOfMemoryError:
            print(f"{name}: out of GPU memory at {frames} frames, {w} x {h}; trying smaller")
            gc.collect()
            torch.cuda.empty_cache()
    else:
        print(f"{name}: could not make a clip, even the smallest size ran out of memory")
print("done:", made)

In [ ]:
#@title 8. Download the clips as one zip (needed when not using Drive)
import shutil
from google.colab import files
shutil.make_archive("/content/clips", "zip", OUT)
files.download("/content/clips.zip")

## After the test clip

- Download `clay-clips/clips/<name>.mp4` from Drive and watch it. If the motion is good, set `FIRST_N` to `0` in cell 3 and **run cells 3, 4, 5 and 7 again** (the model in cell 6 stays loaded unless the session was reset; if it was, run 6 as well).
- **Too slow?** Lower `STEPS` (20), use `SIZE` = draft, or `SECONDS` = 3.
- **Motion too small or wrong?** Edit that picture's line in `prompts.json` (describe what should move, for example "the flames flicker, smoke rises from the left") and delete its old clip from the `clips` folder before running again.
- **Session ended?** Run all again: pictures that already have a clip are skipped.